In [1]:
from bs4 import BeautifulSoup
import pandas as pd
from transformers import AutoConfig
import glob

In [2]:
from vllm import LLM, SamplingParams

In [3]:
# Initialize the model

model = LLM("/mnt/isilon/wang_lab/shared/Llama3_3/Llama-3.3-70B-Instruct/",
    tensor_parallel_size=4, #change it to number of GPU
    max_model_len=5000,
    swap_space=0,
    gpu_memory_utilization = 0.95,
    trust_remote_code=False
)

INFO 06-05 22:09:13 config.py:905] Defaulting to use mp for distributed inference
INFO 06-05 22:09:19 llm_engine.py:237] Initializing an LLM engine (v0.6.3.post1) with config: model='/mnt/isilon/wang_lab/shared/Llama3_3/Llama-3.3-70B-Instruct/', speculative_config=None, tokenizer='/mnt/isilon/wang_lab/shared/Llama3_3/Llama-3.3-70B-Instruct/', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=4, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_tim

Loading safetensors checkpoint shards:   0% Completed | 0/30 [00:00<?, ?it/s]


INFO 06-05 22:16:16 model_runner.py:1067] Loading model weights took 32.8892 GB
(VllmWorkerProcess pid=2273175) (VllmWorkerProcess pid=2273177) (VllmWorkerProcess pid=2273176) INFO 06-05 22:16:16 model_runner.py:1067] Loading model weights took 32.8892 GB
INFO 06-05 22:16:16 model_runner.py:1067] Loading model weights took 32.8892 GB
INFO 06-05 22:16:16 model_runner.py:1067] Loading model weights took 32.8892 GB
INFO 06-05 22:16:24 distributed_gpu_executor.py:57] # GPU blocks: 766, # CPU blocks: 0
INFO 06-05 22:16:24 distributed_gpu_executor.py:61] Maximum concurrency for 5000 tokens per request: 2.45x
INFO 06-05 22:16:24 model_runner.py:1395] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 06-05 22:16:24 model_runner.py:1399] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreas

In [6]:
import re
import os
# Load train corpus 
with open('../../LLM/tmVar/tmVarCorpus/train.PubTator.txt', 'r') as file:
    corpus = file.read()
# Regular expression to match PubMed ID and the section after "|a|"
pattern = r'(\d+)\|a\|([^\n]+)'
# Find all matches
matches = re.findall(pattern, corpus)
parsed_dict = {match[0]: match[1].strip() for match in matches}

tmvar_corpus=[]
pmid_list=[]
for pubmed_id, abstract in parsed_dict.items():
    pmid_list.append(pubmed_id)
    tmvar_corpus.append(f"Abstract: {abstract}\n")
    #print(f"PubMed ID: {pubmed_id}\nAbstract: {abstract}\n")

# Load test corpus
with open('../../LLM/tmVar/tmVarCorpus/test.PubTator.txt', 'r') as file:
    corpus = file.read()
    
matches2 = re.findall(pattern, corpus)
parsed_dict2 = {match[0]: match[1].strip() for match in matches2}

for pubmed_id, abstract in parsed_dict2.items():
    pmid_list.append(pubmed_id)
    tmvar_corpus.append(f"Abstract: {abstract}\n")
    #print(f"PubMed ID: {pubmed_id}\nAbstract: {abstract}\n")


In [7]:
len(tmvar_corpus)

500

In [9]:
tmvar_corpus[0]

'Abstract: Factor XI (FXI) deficiency is a rare autosomal recessive coagulation disorder most commonly found in Ashkenazi and Iraqi Jews, but it is also found in other ethnic groups. It is a trauma or surgery-related bleeding disorder, but spontaneous bleeding is rarely seen. The clinical manifestation of bleeding in FXI deficiency cases is variable and seems to poorly correlate with plasma FXI levels. The molecular pathology of FXI deficiency is mutation in the F11 gene on the chromosome band 4q35. We report a novel mutation of the F11 gene in an 18-year-old asymptomatic Korean woman with mild FXI deficiency. Pre-operative laboratory screen tests for lipoma on her back revealed slightly prolonged activated partial thromboplastin time (45.2 sec; reference range, 23.2-39.4 sec). Her FXI activity (35%) was slightly lower than the normal FXI activity (reference range, 50-150%). Direct sequence analysis of the F11 gene revealed a heterozygous A to G substitution in nucleotide 1517 (c.1517A

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("/mnt/isilon/wang_lab/shared/Llama3_3/Llama-3.3-70B-Instruct")


In [16]:
def generate_prompt(tmvar_corpus, type_of_prompt):
    '''
    type_of_prompt:'zeroshot_prompt', 'cot_para_prompt', 'para_cot_prompt', 'fewshot_prompt', 'cot_fewshot_prompt'
    '''
    prompts= []
    for paragraph in tmvar_corpus:
        # Combine CoT prompt with the actual paragraph
        if type_of_prompt=='zeroshot_prompt':
            prompt=f'{zero_shot_prompt}\nParagraph: \"{paragraph}\"'
        elif type_of_prompt=='cot_para_prompt':
            prompt=f'{cot_prompt_final}\nParagraph: \"{paragraph}\"'
        elif type_of_prompt=='para_cot_prompt':
            prompt=f'Paragraph: \"{paragraph}\"\n{cot_prompt_final}'
        elif type_of_prompt=='fewshot_prompt':
            prompt=f'{few_shot_examples}\nParagraph: \"{paragraph}\"'
        elif type_of_prompt=='cot_fewshot_prompt':
            prompt=f'{cot_prompt_final}\n{few_shot_examples}\nParagraph: \"{paragraph}\"'

        #short system prompt
        messages = [
            {'role': 'system', 'content': 
             """You are an expert in genomics and bioinformatics. Extract the variant-related information from scientific publication paragraphs."""},
            {'role': 'user', 'content': prompt},  # Instruction (CoT)
        ]

        # Apply chat template
        templated_prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,  # Adds necessary tokens for generation
            tokenize=False               # Returns a string
        )

        templated_prompt=templated_prompt.replace('Cutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n','')
        prompts.append(templated_prompt)
    
    return prompts

In [29]:
# Chain of Thought (CoT) Prompt
cot_prompt_final = """
Extract the information of gene, DNA mutation, protein mutation, related disease, experiment result, and pathogenicity from this research paper paragraph in the format of ##gene, DNA mutation, protein mutation, disease, experimental result, pathogenicity.

Follow these steps to ensure accuracy:
1. Identify Relevant Genes and Mutations, locate any gene names or symbols mentioned in the paper.
2. Identify if there is any DNA-level mutation. Make sure the DNA mutation only contains letter A,T,C,G. Note any variant start with 'c.'.
3. Identify if there is any protein-level mutation. Note different expression of a protein or amino acid, no matter in three letter or one letter. If only an RSID is found, fill it in as the DNA mutation. Note any variant start with 'p.'.
4. Determine Disease Association: identify any diseases or conditions related to each gene and mutation. Look for terms like "disease," "syndrome," "condition," or names of specific conditions.
5. Extract Functional or Experimental Result: find statements describing the effect of each mutation, especially in terms of experimental assays or functional impacts. Note results like "rescued" or "failed to rescue" a function.
6. Assign Pathogenicity Level: determine if the mutation is "pathogenic," "benign," or similar terms indicating pathogenicity or clinical relevance.
7. Format Output Concisely: present the result in this format: ##gene, DNA mutation, protein mutation, disease, experimental result, pathogenicity.

Using these steps, extract and present the unique ##gene, DNA mutation, protein mutation, disease, experiment result, pathogenicity for each mutation-disease relationship in this research paper paragraph/abstract:
"""
cot_prompt = """Extract the information of gene, DNA mutation, protein mutation, related disease, experiment result, and pathogenicity from this research paper paragraph in the format of ##gene, DNA mutation, protein mutation, disease, experimental result, pathogenicity. Follow these steps to ensure accuracy:

1. Identify Relevant Genes and Mutations, locate any gene names or symbols mentioned in the paper.
2. Identify if there is any DNA-level mutation. Make sure the DNA mutation only contains letter A,T,C,G. Note any variant start with 'c.'. 
3. Identify if there is any protein-level mutation. Note different expression of a protein or amino acid, no matter in three letter or one letter. If only a RSID is found, fill it in as the DNA mutation. Note any variant start with 'p.'. 
4. Determine Disease Association: identify any diseases or conditions related to each gene and mutation. Look for terms like "disease," "syndrome," "condition," or names of specific conditions.
5. Extract Functional or Experimental Result: find statements describing the effect of each mutation, especially in terms of experimental assays or functional impacts. Note results like "rescued" or "failed to rescue" a function.
6. Assign Pathogenicity Level: determine if the mutation is "pathogenic," "benign," or similar terms indicating pathogenicity or clinical relevance.
7. Format Output Concisely: present the result in this format: ##gene, DNA mutation, protein mutation, disease, experimental result, pathogenicity

Using these steps, extract and present the unique ##gene, DNA mutation, protein mutation, disease, experiment result, pathogenicity for each mutation-disease relationship in this research paper only once. Stop respond when you find all mutations.
"""
cot_instruction = """
Extract gene, DNA mutation, protein mutation, disease, experimental result, and pathogenicity from the paragraph. Follow these steps:
1. Identify genes and mutations (DNA with A, T, C, G, e.g., 'c.123A>T', protein with amino acids like 'p.Glu6Val').
2. Note associated diseases and functional results.
3. Assign pathogenicity (e.g., 'pathogenic', 'benign').
4. Format: ##gene, DNA mutation, protein mutation, disease, experimental result, pathogenicity.
"""
few_shot_examples = """
Extract the information of gene, DNA mutation, protein mutation, related disease, experiment result, and pathogenicity from this research paper paragraph. Answer exactly in the format of ##gene, DNA mutation, protein mutation, disease, experimental result, pathogenicity.

Example 1:
Paragraph: "Paragraph: For ESRP2, variants R250Q, R353Q and R667C rescued the molecular splicing of Arhgef11 in the Py2T assay, (Fig. 3E, Table 1)."
Output: 
##ESRP2, -, R250Q, orofacial cleft (OFC), rescued the molecular splicing of Arhgef11 in the Py2T assay, benign
##ESRP2, -, R353Q, orofacial cleft (OFC), rescued the molecular splicing of Arhgef11 in the Py2T assay, benign
##ESRP2, -, R667C, orofacial cleft (OFC), rescued the molecular splicing of Arhgef11 in the Py2T assay, benign

Example 2:
Paragraph: "The mutation c.123A>T in BRCA1 leads to a truncated protein p.Lys41*, associated with breast cancer. Functional assays showed loss of DNA repair function, indicating pathogenicity."
Output: 
##BRCA1, c.123A>T, p.Lys41*, breast cancer, loss of DNA repair function, pathogenic

Example 3:
Paragraph: "A variant in CFTR, p.Phe508del, causes cystic fibrosis. Studies confirmed defective chloride channels, confirming pathogenicity."
Output: 
##CFTR, -, p.Phe508del, cystic fibrosis, defective chloride channels, pathogenic
"""
zero_shot_prompt="""
Extract the information of gene, DNA mutation, protein mutation, related disease, experiment result, and pathogenicity from this research paper paragraph in the format of ##gene, DNA mutation, protein mutation, disease, experimental result, pathogenicity.
"""

# Loop over the paragraphs
zeroshot_prompts = generate_prompt(tmvar_corpus,"zeroshot_prompt")
cot_para_prompts = generate_prompt(tmvar_corpus,"cot_para_prompt")
para_cot_prompts = generate_prompt(tmvar_corpus,"para_cot_prompt")
fewshot_prompts = generate_prompt(tmvar_corpus,"fewshot_prompt")
cot_fewshot_prompts = generate_prompt(tmvar_corpus,"cot_fewshot_prompt")

# Example output
print(zeroshot_prompts[0])


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert in genomics and bioinformatics. Extract the variant-related information from scientific publication paragraphs.<|eot_id|><|start_header_id|>user<|end_header_id|>

Extract the information of gene, DNA mutation, protein mutation, related disease, experiment result, and pathogenicity from this research paper paragraph in the format of ##gene, DNA mutation, protein mutation, disease, experimental result, pathogenicity.

Paragraph: "Abstract: Factor XI (FXI) deficiency is a rare autosomal recessive coagulation disorder most commonly found in Ashkenazi and Iraqi Jews, but it is also found in other ethnic groups. It is a trauma or surgery-related bleeding disorder, but spontaneous bleeding is rarely seen. The clinical manifestation of bleeding in FXI deficiency cases is variable and seems to poorly correlate with plasma FXI levels. The molecular pathology of FXI deficiency is mutation in the F11 gene on the chromos

In [30]:
query_test=fewshot_prompts

### This controls the sampling paramters of the model. Temperature coresponse to how "creative" the model's output should be
sampling_params = SamplingParams(temperature=0.6, #0.7
                                 top_p=0.3, #0.3#use lower value to let model stick to the original text
                                 max_tokens=1000)

#The model's generate method accepts a list of strings:
output_test = model.generate(query_test,
                        sampling_params)

# Initialize an empty list to hold the rows
rows = []
i=0

# Process each line in the output
for x in output_test:

    for line in x.outputs[0].text.strip().splitlines():
        if line.startswith("##"):  # Check if the line starts with '##'
            # Remove the leading '##' and split by comma
            parts = line[2:].split(", ") #use ', ' for old prompt
            rows.append([pmid_list[i]]+parts)
    i+=1

df_cot = pd.DataFrame(rows)
df_cot=df_cot.iloc[:,:7]
df_cot.columns=['PMID','gene', 'DNA mutation', 'protein mutation', 'disease', 'experiment result', 'pathogenicity']

# get the tmVar result, to check the mutation extraction
with open('../../LLM/tmVar/tmVarCorpus/train.PubTator.txt', 'r') as file:
    corpus = file.readlines()

# Regular expression to identify rows starting with a PubMed ID followed by tab-separated values
pattern = r'^\d+\t.+'

# Filter rows matching the pattern
table_rows = [line.strip() for line in corpus if re.match(pattern, line)]
data = [row.split('\t') for row in table_rows]
df1 = pd.DataFrame(data)

# get the tmVar result for test corpus
with open('../../LLM/tmVar/tmVarCorpus/test.PubTator.txt', 'r') as file:
    corpus = file.readlines()

table_rows = [line.strip() for line in corpus if re.match(pattern, line)]
data = [row.split('\t') for row in table_rows]
df2 = pd.DataFrame(data)

#combine train and test
df = pd.concat([df1,df2], ignore_index=True, sort=False)

df_tmVar=df[[0,3,4]]
df_tmVar.columns=['PMID','Mutation','Type']

df_tmVar_dna=df_tmVar[df_tmVar['Type']=='DNAMutation']
df_tmVar_pro=df_tmVar[df_tmVar['Type']=='ProteinMutation']
df_cot=df_cot.drop_duplicates()
df_cot=df_cot[(df_cot['DNA mutation'].str.contains(r'\d', na=False)) |
      (df_cot['protein mutation'].str.contains(r'\d', na=False))] #show the rows that it has number

df_cot_dna=df_cot[['PMID','DNA mutation']].drop_duplicates()
df_cot_pro=df_cot[['PMID','protein mutation']].drop_duplicates()


#####DNA#####
df_cot_dna['DNA mutation']=df_cot_dna['DNA mutation'].apply(lambda x: x[2:] if x[:2]=='c.'else x)
df_tmVar_dna.loc[:,'Mutation']=df_tmVar_dna['Mutation'].apply(lambda x: x[2:] if x[:2]==r'c.'else x)
df_cot_dna=df_cot_dna[df_cot_dna['DNA mutation'].str.contains(r'\d', na=False)]
df_tmVar_dna=df_tmVar_dna[df_tmVar_dna['Mutation'].str.contains(r'\d', na=False)]
#remove the space, _, () for better comparison
df_cot_dna['DNA mutation']=df_cot_dna['DNA mutation'].str.replace(r"[ _()-]", "", regex=True)
df_tmVar_dna['Mutation']=df_tmVar_dna['Mutation'].str.replace(r"[ _()-]", "", regex=True)
df_cot_dna['DNA mutation']=df_cot_dna['DNA mutation'].str.replace(r"incDNA", "", regex=True)
df_cot_dna['DNA mutation']=df_cot_dna['DNA mutation'].str.replace(r"by", "", regex=True)
df_cot_dna['DNA mutation']=df_cot_dna['DNA mutation'].str.replace(r"promoter", "", regex=True)
df_tmVar_dna['Mutation']=df_tmVar_dna['Mutation'].str.replace(r"promoter", "", regex=True)
df_tmVar_dna['Mutation']=df_tmVar_dna['Mutation'].str.replace(r"by", "", regex=True)
df_tmVar_dna['Mutation']=df_tmVar_dna['Mutation'].str.replace(r"incDNA", "", regex=True)



Processed prompts:  67%|██████▋   | 337/500 [04:09<01:19,  2.06it/s, est. speed input: 1028.07 toks/s, output: 152.87 toks/s]

WARNING 06-05 22:54:20 scheduler.py:1483] Sequence group 1350 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=101


Processed prompts: 100%|██████████| 500/500 [06:29<00:00,  1.28it/s, est. speed input: 972.76 toks/s, output: 157.03 toks/s] 


In [42]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [34]:
# DNA
# Group mutations by PMID in both DataFrames
tmVar_grouped = df_tmVar_dna.groupby('PMID')['Mutation'].apply(set).reset_index(name='tmVar_Mutations')
cot_grouped = df_cot_dna.groupby('PMID')['DNA mutation'].apply(set).reset_index(name='cot_Mutations')

# Merge the two grouped DataFrames on PMID
merged = pd.merge(tmVar_grouped, cot_grouped, on='PMID', how='outer')

# Calculate the number of matches and total mutations
'''merged['Matched'] = merged.apply(
    lambda row: len(row['tmVar_Mutations'] & row['cot_Mutations']) if pd.notnull(row['tmVar_Mutations']) and pd.notnull(row['cot_Mutations']) else 0,
    axis=1
)'''
#consider true if the tm_var is a substring of cot_mut
##e.g., 'G/C' in tmVar vs. 'G/C polymorphism at position 135' in pmc-llm
##sometimes the LLM result is better than the tmVar, like the example show above
merged['Matched'] = merged.apply(
    lambda row: sum(
        any(tm_var in cot_mut for cot_mut in row['cot_Mutations']) 
        for tm_var in row['tmVar_Mutations']
    ) if pd.notnull(row['tmVar_Mutations']) and pd.notnull(row['cot_Mutations']) else 0,
    axis=1
)
merged['Total_tmVar'] = merged['tmVar_Mutations'].apply(lambda x: len(x) if pd.notnull(x) else 0)
merged['Total_cot'] = merged['cot_Mutations'].apply(lambda x: len(x) if pd.notnull(x) else 0)

# Drop unnecessary columns if desired
result = merged[['PMID', 'Matched', 'Total_tmVar', 'Total_cot']]

# Display the result
#print(result)
print('##Num of Matched, Num of tmVar result, Accuracy of LLM extraction of DNA mutation: ',
      result.sum()['Matched'],
      result.sum()['Total_tmVar'],
      result.sum()['Matched']/result.sum()['Total_tmVar'])



##Num of Matched, Num of tmVar result, Accuracy of LLM extraction of DNA mutation:  383 438 0.8744292237442922


In [36]:
merged_dna = merged

In [37]:
merged_dna.to_csv('few_shot_benchmark.tsv',sep='\t')

In [43]:
#######protein
#format the amino acid code
# provide 3 letter aa to 1 letter
one_to_three_letter_map = {
    'A': 'ALA', 'R': 'ARG', 'N': 'ASN', 'D': 'ASP',
    'C': 'CYS', 'E': 'GLU', 'Q': 'GLN', 'G': 'GLY',
    'H': 'HIS', 'I': 'ILE', 'L': 'LEU', 'K': 'LYS',
    'M': 'MET', 'F': 'PHE', 'P': 'PRO', 'S': 'SER',
    'T': 'THR', 'W': 'TRP', 'Y': 'TYR', 'V': 'VAL'
}

new_three_to_one_letter_map = {v.capitalize(): k for k, v in one_to_three_letter_map.items()}

df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].astype(str)
df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].apply(lambda x: x[2:] if x[:2]=='p.'else x)
df_tmVar_pro.loc[:, 'Mutation']=df_tmVar_pro['Mutation'].apply(lambda x: x[2:] if x[:2]==r'p.'else x)
df_cot_pro=df_cot_pro[df_cot_pro['protein mutation'].str.contains(r'\d', na=False)]
df_tmVar_pro=df_tmVar_pro[df_tmVar_pro['Mutation'].str.contains(r'\d', na=False)]

#turn 3 letter into 1 letter (not the opposite because there will be issue for words with a upper letter)
# Function to replace three-letter codes with one-letter codes in a string
def replace_amino_acids(text):
    def replace_match(match):
        return new_three_to_one_letter_map.get(match.group(0), match.group(0))

    # Replace three-letter amino acid codes found in the string
    # should allow wrong upper/lower case spelling, e.g., {R753Q, R753GLn}	{R753Q}
    return re.sub(r'[A-Za-z]{3}', replace_match, text)

df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].apply(replace_amino_acids)
df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].apply(replace_amino_acids)

#remove the space, _, () for better comparison
df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].str.replace(r"[ _()-]", "", regex=True)
df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"[ _()-]", "", regex=True)
#remove the 'to', 'by' and replace 'stop' as '*'
df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].str.replace(r"to", "", regex=True)
df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].str.replace(r"stop", "*", regex=True)
df_cot_pro['protein mutation']=df_cot_pro['protein mutation'].str.replace(r"by", "", regex=True)
df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"by", "", regex=True)
df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"to", "", regex=True)
df_tmVar_pro['Mutation']=df_tmVar_pro['Mutation'].str.replace(r"stop", "*", regex=True)


# protein
# Group mutations by PMID in both DataFrames
tmVar_grouped = df_tmVar_pro.groupby('PMID')['Mutation'].apply(set).reset_index(name='tmVar_Mutations')
cot_grouped = df_cot_pro.groupby('PMID')['protein mutation'].apply(set).reset_index(name='cot_Mutations')

# Merge the two grouped DataFrames on PMID
merged = pd.merge(tmVar_grouped, cot_grouped, on='PMID', how='outer')

# Calculate the number of matches and total mutations
'''merged['Matched'] = merged.apply(
    lambda row: len(row['tmVar_Mutations'] & row['cot_Mutations']) if pd.notnull(row['tmVar_Mutations']) and pd.notnull(row['cot_Mutations']) else 0,
    axis=1
)'''
merged['Matched'] = merged.apply(
    lambda row: sum(
        any(str(tm_var) in str(cot_mut) for cot_mut in row['cot_Mutations']) 
        for tm_var in row['tmVar_Mutations']
    ) if pd.notnull(row['tmVar_Mutations']) and pd.notnull(row['cot_Mutations']) else 0,
    axis=1
)
merged['Total_tmVar'] = merged['tmVar_Mutations'].apply(lambda x: len(x) if pd.notnull(x) else 0)
merged['Total_cot'] = merged['cot_Mutations'].apply(lambda x: len(x) if pd.notnull(x) else 0)

# Drop unnecessary columns if desired
result = merged[['PMID', 'Matched', 'Total_tmVar', 'Total_cot']]

# Display the result
#print(result)
print('##Num of Matched, Num of tmVar result, Accuracy of LLM extraction of protein mutation: ',
      result.sum()['Matched'],
      result.sum()['Total_tmVar'],
      result.sum()['Matched']/result.sum()['Total_tmVar'])


##Num of Matched, Num of tmVar result, Accuracy of LLM extraction of protein mutation:  328 376 0.8723404255319149


In [40]:
merged_protein = merged

In [41]:
merged_protein.to_csv('few_shot_benchmark.protein.tsv',sep='\t')

In [45]:
merged_dna[merged_dna['Total_tmVar']>merged_dna['Matched']]

,PMID,tmVar_Mutations,cot_Mutations,Matched,Total_tmVar,Total_cot
8,12701064,{TAC>AAatcodon329},"{1basedeletionatcodon329TAC>AA, 3basedeletionTCC}",0,1,2
13,12737948,"{IVS10+1,g>t, 251G, IVS348C, IVS123T}","{IVS348C, IVS10+1, 251G, IVS123T}",3,4,4
15,12752111,"{IVSI129A>C, IVSI1G>A, IVSI5G>C, codonCD26GAG>GAA}","{IVSI5G>C, alpha4.2deletion, IVSI129A>C, 13bpdeletionCD6toCD10, alpha3.7deletion, CD26GAG>GAA, CD55A, IVSI1G>A}",3,4,8
27,14510914,{del439443},{13141328delins15nt},0,1,1
37,14722925,"{87+1G>A, G>Asubstitutionatnucleotide+1ofintron2, 27delC}","{N34S, 87+1G>A, 27delC}",2,3,3
39,14751036,"{2068G>C, 9+1G>T, 1242G>T, 2084G>A, 1780C>A}","{GGCCinsertionatexon8, Cinsertioninexon14betweenpositions25052511, 2068G>C, 1G>T, 1242G>T, 2084G>A, 1780C>A}",4,5,7
51,15111599,"{279delGAG, IVS3+1G/A}",{279delGAG},1,2,1
63,15316799,"{intron2,352G>A, intron4,+718C>T, 15T>C, intron3,+45C>T}","{352G>A, +718C>T, +45C>T, 15T>C}",1,4,4
68,15481887,"{codon29,GGC>AGC, codon26,GAG>GCG}",NaN,0,2,0
69,15485686,{G>Asubstitutionatcodon1763},{G1763A},0,1,1


In [46]:
merged_protein[merged_protein['Total_tmVar']>merged_protein['Matched']]

,PMID,tmVar_Mutations,cot_Mutations,Matched,Total_tmVar,Total_cot
9,12898858,{M235T},NaN,0,1,0
22,14722925,{N34S},NaN,0,1,0
23,14742428,"{S133sp, C219W, H391R, V69G, H156Y}","{C219W, H391R, V69G, H156Y, S133*}",4,5,5
26,14979495,"{R753Q, R753GLn}",NaN,0,2,0
32,15111599,{DeltaG91},{G91del},0,1,1
33,15122711,"{E873Sp, A467T}","{E873*, A467T}",1,2,2
43,15316799,{F>Satcodon15},{F15S},0,1,1
47,15485686,"{I1762A, V1763M, V1764M}",{V1763M},1,3,1
50,15623763,"{R124C, leucine269phenylalanine, valine539aspartic, arginine124cysteine, glycine594valine, H626R, L269F, V539D, histidine626arginine, V624V625del, R124L, G594V, arginine555trypphan, R555W}","{R124C, L269F, H626R, V539D, V624V625del, R124L, G594V, R555W}",8,14,8
63,15867855,{157delMTTTVP},NaN,0,1,0


In [28]:
# loop over the result output
i=0
for x in output_test:
    if pmid_list[i] == '21325775':
        print("="*30,pmid_list[i],"="*30)
        print(x.outputs[0].text)
        print("="*30,"="*30)
        print(tmvar_corpus[i])
    i+=1

============================== 21325775 ==============================
##APOE||-||p.Pro158Arg||Lipoprotein glomerulopathy (LPG)||nephrotic syndrome, mild dyslipidemia||conformational change of apoE protein and affects the interaction between abnormal apoE-containing lipoproteins and the endothelial cells of glomerular capillaries||likely pathogenic
============================== ==============================
Abstract: Lipoprotein glomerulopathy (LPG) is a rare disease characterized by the presence of thrombuslike deposition in markedly dilated glomerular capillaries and is often accompanied by an increased serum apolipoprotein E (apoE) level. Several gene mutations of apoE have been reported to be associated with LPG. In the current study, we report an LPG patient with a novel apoE mutation, apoE Osaka. The patient was a 45-year-old man who was hospitalized due to nephrotic syndrome. Light and electron microscopic observations of renal biopsy clearly showed characteristic findings of 

In [ ]:
tmvar_corpus[]